# TaskAug: Task-Adaptive Data Augmentation for ECG Classification

**Paper:** Raghu et al. (2022). *Data Augmentation for Electrocardiograms.* CHIL 2022.  
**Link:** https://proceedings.mlr.press/v174/raghu22a.html  
**Authors:** Paul Garcia (alanpg2) · Rogelio Medina (orm9) · Cesar Nava (can14)

This notebook demonstrates the full pipeline:
1. Install dependencies
2. Clone the PyHealth PR branch
3. Generate synthetic ECG data
4. Train with standard and bi-level optimisation
5. Run the 4-configuration ablation study
6. Display results

## 1 · Install Dependencies

> ⚠️ **This cell will automatically restart the kernel at the end.**  
> After it reconnects, run **Cell 2 onward** — do NOT re-run Cell 1.

In [ ]:
import subprocess, sys

# Colab ships numpy 2.x; pyhealth requires numpy <2.0 — force-reinstall first
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "numpy==1.26.4", "--force-reinstall", "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "wfdb", "pyhealth", "-q"])

print("✅ Installation complete — restarting kernel now...")
import os, signal
os.kill(os.getpid(), signal.SIGKILL)  # forces kernel restart; Colab reconnects automatically

## 2 · Clone Repo and Set Up Path

In [ ]:
import os, sys

REPO = '/content/pyhealth-dl4h-sp26'

if not os.path.exists(REPO):
    !git clone -b taskaug-ecg https://github.com/paulgarciaro/pyhealth-dl4h-sp26.git
else:
    !git -C {REPO} pull

# Add repo root and examples/ to path so imports resolve
for p in [REPO, os.path.join(REPO, 'examples')]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(os.path.join(REPO, 'examples'))
print('Working directory:', os.getcwd())

## 3 · Verify Imports

In [ ]:
import torch
from ptbxl_ecg_classification_taskaug_resnet import (
    make_synthetic_dataset,
    train_standard,
    BiLevelTrainer,
    run_ablation,
    evaluate,
)
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

## 4 · Generate Synthetic Data

200 training / 50 validation samples.  
Positive-class signals have a 5 Hz sine wave injected into lead 0 — a learnable discriminative signal.

In [ ]:
train_ds, val_ds = make_synthetic_dataset(n_train=200, n_val=50, seed=42)

x, y = train_ds[0]
print(f'ECG shape : {x.shape}   (leads × time steps)')
print(f'Label     : {y.item()}')
print(f'Train size: {len(train_ds)}   Val size: {len(val_ds)}')

## 5 · Standard Training (joint Adam on backbone + policy)

In [ ]:
from unittest.mock import MagicMock
from pyhealth.models.taskaug_resnet import TaskAugResNet

def mock_dataset():
    ds = MagicMock()
    ds.input_schema = {'ecg': 'tensor'}
    ds.output_schema = {'label': 'binary'}
    proc = MagicMock()
    proc.size.return_value = 1
    ds.output_processors = {'label': proc}
    return ds

EPOCHS      = 10
BATCH_SIZE  = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model_std = TaskAugResNet(mock_dataset(), policy_stages=2)
history_std = train_standard(
    model_std, train_loader, val_loader,
    epochs=EPOCHS, lr=1e-3, device=DEVICE,
)

## 6 · Bi-Level Training (DARTS first-order approximation)

Inner loop updates the ResNet backbone on augmented training data.  
Outer loop updates the policy on clean validation loss.

In [ ]:
model_bl = TaskAugResNet(mock_dataset(), policy_stages=2)
trainer  = BiLevelTrainer(model_bl, lr_inner=1e-3, lr_outer=1e-2, device=DEVICE)
history_bl = trainer.fit(train_loader, val_loader, epochs=EPOCHS)

## 7 · Standard vs Bi-Level: Final Metrics

In [ ]:
metrics_std = evaluate(model_std, val_loader, DEVICE)
metrics_bl  = evaluate(model_bl,  val_loader, DEVICE)

print(f"{'Method':<20} {'AUROC':>7}  {'Accuracy':>9}")
print('-' * 42)
print(f"{'Standard (joint)':<20} {metrics_std['auroc']:>7.3f}  {metrics_std['accuracy']:>9.3f}")
print(f"{'Bi-Level (DARTS)':<20} {metrics_bl['auroc']:>7.3f}  {metrics_bl['accuracy']:>9.3f}")

## 8 · Ablation Study

Compares four augmentation configurations on the same synthetic split:

| Config | Description |
|--------|-------------|
| A | No augmentation (backbone only) |
| B | Fixed Gaussian noise (σ=0.1) |
| C | TaskAug K=1 (learned policy, 1 stage) |
| D | TaskAug K=2 (learned policy, 2 stages — paper default) |

In [ ]:
run_ablation(train_ds, val_ds, DEVICE, epochs=EPOCHS, batch_size=BATCH_SIZE)

## 9 · Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history_std]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AUROC
axes[0].plot(epochs, [h['auroc']    for h in history_std], 'b-o', label='Standard')
axes[0].plot(epochs, [h['auroc']    for h in history_bl],  'r-s', label='Bi-Level')
axes[0].set_title('Validation AUROC')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUROC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [h['accuracy'] for h in history_std], 'b-o', label='Standard')
axes[1].plot(epochs, [h['accuracy'] for h in history_bl],  'r-s', label='Bi-Level')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('TaskAug: Standard vs Bi-Level Training (Synthetic Data)', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## 10 · (Optional) Real PTB-XL Data

If you have access to PTB-XL, mount Google Drive and point to the data root.

```python
from google.colab import drive
drive.mount('/content/drive')

from ptbxl_ecg_classification_taskaug_resnet import make_ptbxl_dataset
train_ds, val_ds = make_ptbxl_dataset(
    data_root='/content/drive/MyDrive/ptb-xl/',
    task_label='MI',
    sampling_rate=100,
)
```